# Nagelhus et al. (2023), words and counts

**Paper.** Nagelhus et al. (2023), *Neuron* 111, 2091-2104,
[doi:10.1016/j.neuron.2023.04.008](https://doi.org/10.1016/j.neuron.2023.04.008); claims file written
from the version of record with its STAR Methods and Supplemental information.

**Dataset.** EBRAINS [10.25493/Q0FG-1X8](https://doi.org/10.25493/Q0FG-1X8), version v1, the version
whose DOI the paper cites and the only one published (data check in `01_single_cells.ipynb`).

| Claim | Tested or blocked |
|---|---|
| fig5b-words-more-informative | tested |
| fig5bc-words-information-falls-with-distance | tested |
| fig5b-counts-uninformative | tested |
| fig5e-kl-divergence | tested |
| fig5-entropy | tested |
| figs7cd-number-of-words | tested |
| figs7e-downsampled-words | tested |

Spike counts in 10 ms bins over each trial of the 16 decoding sessions (Methods, Definition of words
and counts): tetrode sessions with all their cells, Neuropixels sessions subsampled to 30 cells,
under the claims file's readings (a) one random subset and (b) the mean over 20 subsets. Each 10 ms
bin takes the distance from the object (the Object trial's position) of the nearest position sample,
and each trial's bins are split into 5 equal-sample distance bins. The all-zero word and the zero
count are left out. Mutual information with the trial label, from frequency distributions, minus the
mean over 50 label shuffles. The Neuropixels sessions' tracking comes from the CA1 file (notebook 01).

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import objcode as oc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_columns", 40)

started = time.time()
RAW_POP = []
for p in oc.POP_FILES:
    RAW_POP += oc.load(p)[0]
CA1, _ = oc.load(oc.SINGLE_FILES["CA1"])
POP, replaced = oc.decoding_sessions(RAW_POP, CA1)
del CA1
print(f"dataset {oc.DATASET_ID} version {oc.DATASET_VERSION}: read in {time.time() - started:.0f} s; tracking from the CA1 file for {replaced}")
SESS = [f"{p.rat}#{p.index}" for p in POP]

In [ ]:
rows = []
for k, p in enumerate(POP):
    M, _ = oc.binned_counts(p.trials[0], range(p.n_units))
    rows.append({"session": SESS[k], "cells": p.n_units, "10 ms bins NO": len(M), "max count": int(M.max()),
                 "fraction of bins with a spike": float((M.sum(1) > 0).mean()),
                 "fraction of entries above 1": float((M > 1).mean())})
print(pd.DataFrame(rows).to_string(index=False))

In 10 ms bins nearly all entries are 0 or 1, as the Methods say ("mostly ones and zeroes"). The
tetrode sessions have 25 to 39 cells, as the Methods say.

In [ ]:
def subsets(k, reading):
    p = POP[k]
    if p.technique == "tetrodes":
        return [list(range(p.n_units))]
    rng = np.random.default_rng(oc.seed_for("words-subsets", k))
    n = 1 if reading == "a" else 20
    return [sorted(rng.permutation(p.n_units)[:30]) for _ in range(n)]


def session_measures(k, units, rng):
    """Per distance bin: MI of words and counts (bias-subtracted and raw), KL divergence (and on
    shuffled labels), distinct words per trial, downsampled-word MI; and entropies."""
    p = POP[k]
    obj = p.trials[1].obj
    mats, rings = [], []
    for t in p.trials:
        M, pos = oc.binned_counts(t, units)
        d = np.hypot(pos[:, 0] - obj[0], pos[:, 1] - obj[1])
        rings.append(oc.assign(d, oc.equal_sample_edges(d)))
        mats.append(M)
    wA, wB, zero = oc.word_ids(mats[0], mats[1])
    cA, cB = mats[0].sum(1), mats[1].sum(1)
    out = {"bins": []}
    for b in range(5):
        a_, b_ = rings[0] == b, rings[1] == b
        ww = np.r_[wA[a_], wB[b_]]
        cc = np.r_[cA[a_], cB[b_]]
        lab = np.r_[np.zeros(a_.sum(), int), np.ones(b_.sum(), int)]
        nzw, nzc = ww != zero, cc > 0
        miw, miw_raw = oc.mi_bias_corrected(ww[nzw], lab[nzw], rng)
        mic, mic_raw = oc.mi_bias_corrected(cc[nzc], lab[nzc], rng)
        # KL divergence D(NO || O), over words seen in both trials, probabilities over all timepoints
        def kl(words, labels):
            nA, nB = (labels == 0).sum(), (labels == 1).sum()
            sel = words != zero
            u, inv = np.unique(words[sel], return_inverse=True)
            ca = np.bincount(inv[labels[sel] == 0], minlength=len(u))
            cb = np.bincount(inv[labels[sel] == 1], minlength=len(u))
            both = (ca > 0) & (cb > 0)
            pa, pb = ca[both] / nA, cb[both] / nB
            return float(np.sum(pa * np.log2(pa / pb)))
        kl_true = kl(ww, lab)
        kl_sh = kl(ww, lab[rng.permutation(len(lab))])
        # downsampled words, variant (round 1): each trial keeps its own c most frequent words, so a word
        # kept in one trial only carries the label; shown, not used for the outcome
        keep = np.zeros(len(ww), bool)
        for t in (0, 1):
            m_t = (lab == t) & nzw
            c_t = len(np.unique(cc[(lab == t) & nzc]))
            u, n_ = np.unique(ww[m_t], return_counts=True)
            top = u[np.argsort(-n_)[:c_t]]
            keep |= m_t & np.isin(ww, top)
        mids_v, mids_v_raw = oc.mi_bias_corrected(ww[keep], lab[keep], rng)
        # downsampled words as the criterion says: per session and distance bin, the c most frequent words
        # of both trials together, c the number of distinct counts in the bin (both trials); own generator
        c_bin = len(np.unique(cc[nzc]))
        u, n_ = np.unique(ww[nzw], return_counts=True)
        top = u[np.argsort(-n_, kind="stable")[:c_bin]]
        keep = nzw & np.isin(ww, top)
        rng_ds = np.random.default_rng(oc.seed_for("figs7e-downsampled-words", k, b, *units[:3], len(units)))
        mids, mids_raw = oc.mi_bias_corrected(ww[keep], lab[keep], rng_ds)
        out["bins"].append({"mi words": miw, "mi words raw": miw_raw, "mi counts": mic, "mi counts raw": mic_raw,
                            "kl": kl_true, "kl shuffled": kl_sh, "mi words ds": mids, "mi words ds raw": mids_raw,
                            "mi words ds per trial": mids_v, "mi words ds per trial raw": mids_v_raw, "ds states": c_bin,
                            "ds samples kept": float(keep.sum() / max(nzw.sum(), 1)),
                            "distinct NO": len(np.unique(wA[a_ & (wA != zero)])), "distinct O": len(np.unique(wB[b_ & (wB != zero)])),
                            "H words NO bin": oc.entropy(wA[a_ & (wA != zero)]), "H words O bin": oc.entropy(wB[b_ & (wB != zero)]),
                            "H counts NO bin": oc.entropy(cA[a_ & (cA > 0)]), "H counts O bin": oc.entropy(cB[b_ & (cB > 0)])})
    out["H words a"] = [oc.entropy(w[w != zero]) for w in (wA, wB)]
    out["H words b"] = [oc.entropy(w) for w in (wA, wB)]
    out["H counts a"] = [oc.entropy(c[c > 0]) for c in (cA, cB)]
    out["H counts b"] = [oc.entropy(c) for c in (cA, cB)]
    out["distinct trial"] = [len(np.unique(w[w != zero])) for w in (wA, wB)]
    return out


t0 = time.time()
RES = {}
for reading in "ab":
    per_session = []
    for k in range(len(POP)):
        rng = np.random.default_rng(oc.seed_for("fig5b-words-more-informative", reading, k))
        runs = [session_measures(k, u, rng) for u in subsets(k, reading)]
        # mean over subsets
        bins = [{key: np.mean([r["bins"][b][key] for r in runs]) for key in runs[0]["bins"][b]} for b in range(5)]
        rest = {key: np.mean([r[key] for r in runs], axis=0) for key in runs[0] if key != "bins"}
        per_session.append({"bins": bins, **rest})
    RES[reading] = per_session
    print(f"reading {reading}: {len(per_session)} sessions in {time.time() - t0:.0f} s")


def table(reading, key):
    return np.array([[s["bins"][b][key] for b in range(5)] for s in RES[reading]])

## fig5b-words-more-informative

> Population words (patterns of spiking and silence in 10 ms bins) carried much more information
> about the trial than population spike counts.

In [ ]:
rows = []
for reading in "ab":
    W, C = table(reading, "mi words"), table(reading, "mi counts")
    per_bin = [oc.ranksum(W[:, b], C[:, b]) for b in range(5)]
    pooled = oc.ranksum(W.ravel(), C.ravel())
    rows.append({"subsets": reading, **{f"words bin{b + 1}": np.median(W[:, b]) for b in range(5)},
                 **{f"counts bin{b + 1}": np.median(C[:, b]) for b in range(5)},
                 "(c) per-bin p max": max(x[1] for x in per_bin), "(c) all words higher": bool(np.all(np.median(W, 0) > np.median(C, 0))),
                 "(d) pooled z": pooled[0], "(d) pooled p": pooled[1]})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Words carry more information about the trial than counts in every distance
bin, under both subset readings. Bias-subtracted MI of words has medians 0.075 to 0.156 bits; counts
have 0.0002 to 0.0014 bits. Every per-bin rank-sum test (reading c) and the pooled test (reading d)
give p < 0.05 with words higher (p-values above).

## fig5bc-words-information-falls-with-distance

> The information from words decreased with distance from the object, as decoding did.

In [ ]:
rows = []
for reading in "ab":
    W = table(reading, "mi words")
    h, p = oc.kruskal(W.T)
    med = np.median(W, 0)
    rows.append({"subsets": reading, "H": h, "p": p, **{f"median bin{b + 1}": med[b] for b in range(5)},
                 "bin 1 highest": int(np.argmax(med)) == 0})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** The information from words falls with distance. Kruskal-Wallis H = 11.6 and
13.1 (p = 0.021 and 0.011) for the two subset readings, with bin 1 highest (median 0.155 bits against
0.075 in bin 5).

## fig5b-counts-uninformative

> The information from counts did not depend on distance and was not significantly above zero.

In [ ]:
rows = []
for reading in "ab":
    C = table(reading, "mi counts")
    h, p = oc.kruskal(C.T)
    tt = [stats.ttest_1samp(C[:, b], 0) for b in range(5)]
    rows.append({"subsets": reading, "KW H": h, "KW p": p, **{f"mean bin{b + 1}": C[:, b].mean() for b in range(5)},
                 **{f"t p bin{b + 1}": tt[b].pvalue for b in range(5)}})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** The information from counts does not depend on distance (Kruskal-Wallis
p = 0.19 and 0.29). But it is above zero in four of the five bins by one-sample t tests (p = 0.0039
to 0.050), under both subset readings, while the criterion asks that no bin's test give p < 0.05. The
values are small (means 0.0011 to 0.0035 bits).

## fig5e-kl-divergence

> The KL divergence between the word distributions of NO and O fell with distance from the object,
> from bin 1 to bin 5, while shuffled data showed no decrease.

In [ ]:
rows = []
for reading in "ab":
    K, KS = table(reading, "kl"), table(reading, "kl shuffled")
    h, p = oc.kruskal(K.T)
    w, pw = oc.signrank(K[:, 0], K[:, 4])
    hs, ps = oc.kruskal(KS.T)
    rows.append({"subsets": reading, "H": h, "p": p, "W 1v5": w, "p 1v5": pw, **{f"median bin{b + 1}": np.median(K[:, b]) for b in range(5)},
                 "shuffled H": hs, "shuffled p": ps, **{f"shuffled median bin{b + 1}": np.median(KS[:, b]) for b in range(5)}})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** The KL divergence of NO from O falls with distance: Kruskal-Wallis H = 15.5
and 18.0 (p = 0.0038 and 0.0012), and bin 1 is above bin 5 by signed-rank test (p = 0.0021 and
0.0017; medians 0.169 against 0.050 bits). On shuffled labels there is no effect of distance (p = 0.41
and 0.98).

## fig5-entropy

> The entropy of words was higher than that of counts (1.65 against 0.84 bits).

In [ ]:
rows = []
for reading in "ab":
    R = RES[reading]
    ent = {"a": (np.mean([s["H words a"] for s in R]), np.mean([s["H counts a"] for s in R])),
           "b": (np.mean([s["H words b"] for s in R]), np.mean([s["H counts b"] for s in R])),
           "c": (np.mean([[s["bins"][b][f"H words {t} bin"] for b in range(5) for t in ("NO", "O")] for s in R]),
                 np.mean([[s["bins"][b][f"H counts {t} bin"] for b in range(5) for t in ("NO", "O")] for s in R]))}
    for unit, (hw, hc) in ent.items():
        rows.append({"subsets": reading, "unit": unit, "words": hw, "counts": hc,
                     "words ok": abs(hw - 1.65) <= 0.165, "counts ok": abs(hc - 0.84) <= 0.084})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** Word entropy is 5.5 to 5.9 bits with the all-zero word left out
(readings a and c), and 1.90 to 1.97 bits with it included (b), against 1.65 ± 0.165. Count
entropy is 1.07 to 1.08 bits (a, c) and 0.92 to 0.93 bits (b), against 0.84 ± 0.084. Only one value
is within tolerance (counts, reading b, 20 subsets: 0.919). Words have the higher entropy in every
reading, as the claim says.

## figs7cd-number-of-words

> The number of distinct words did not differ between trial types or across distance bins.

Rank-sum test of NO against O, which the legend runs over "n = 16 experiments". The criterion now
names the readings (answered 2026-10-21: the paper does not say): (w-a) the number of distinct words
of each session's whole trial, 16 against 16; (w-b) the 80 session and distance bin values of each
trial.

In [ ]:
rows = []
for reading in "ab":
    DN, DO = table(reading, "distinct NO"), table(reading, "distinct O")
    hN, pN = oc.kruskal(DN.T)
    hO, pO = oc.kruskal(DO.T)
    z1, p1 = oc.ranksum(DN.ravel(), DO.ravel())
    tr = np.array([s["distinct trial"] for s in RES[reading]])
    z2, p2 = oc.ranksum(tr[:, 0], tr[:, 1])
    rows.append({"subsets": reading, "KW NO H": hN, "KW NO p": pN, "KW O H": hO, "KW O p": pO,
                 "w-a z": z2, "w-a p": p2, "w-b z": z1, "w-b p": p1,
                 "median distinct NO (bin)": np.median(DN), "median distinct O (bin)": np.median(DO)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** The number of distinct words does not differ across distance bins (NO
p = 0.91 and 0.90, O p = 0.78) or between NO and O (w-a p = 0.94, w-b p = 0.85), under both subset
readings.

## figs7e-downsampled-words

> After downsampling words to as many states as counts, words still carried more information than
> counts.

In [ ]:
rows = []
for selection, key in (("per session and bin (criterion)", "mi words ds"), ("per trial (variant, leaks the label)", "mi words ds per trial")):
    for reading in "ab":
        for bias in ("a", "b"):
            WD = table(reading, key if bias == "a" else key + " raw")
            C = table(reading, "mi counts" if bias == "a" else "mi counts raw")
            W_full = table(reading, "mi words" if bias == "a" else "mi words raw")
            w, p = oc.signrank(WD.ravel(), C.ravel())
            rows.append({"word selection": selection, "subsets": reading, "bias subtracted": bias == "a", "W": w, "p": p,
                         "median words ds": np.median(WD), "median all words": np.median(W_full), "median counts": np.median(C),
                         "pairs with words higher": int((WD.ravel() > C.ravel()).sum()), "pairs": WD.size})
print(pd.DataFrame(rows).to_string(index=False))
print()
for reading in "ab":
    S = table(reading, "ds states")
    F = table(reading, "ds samples kept")
    print(f"states kept per session and bin (subsets {reading}): {S.min():.0f} to {S.max():.0f}; "
          f"fraction of non-zero word samples kept: {F.min():.3f} to {F.max():.3f} (median {np.median(F):.3f})")
print(f"\nTotal time {time.time() - started:.0f} s")


**Result: reproduced.** After downsampling as the criterion says (per session and distance bin, the
c most frequent words of both trials together, c the number of distinct counts there, 5 to 16 under subsets a and 5 to 18 under subsets b),
words are more informative than counts in all 80 session-bin pairs, with signed-rank W = 0,
p = 7.9e-15, with or without the bias subtraction and under both subset readings. Downsampled words
give a median of 0.053 to 0.064 bits against 0.0007 to 0.0013 bits for counts, and less than all
words (0.096 to 0.148 bits). The selection keeps 31 to 69% of the non-zero word samples under both subset readings. The earlier
selection, each trial keeping its own most frequent words, is shown as a variant: it gives 0.30 bits,
more than all words, because a word kept in one trial only carries the label; it does not decide the
outcome.


## Summary

| Claim | Outcome |
|---|---|
| fig5b-words-more-informative | reproduced |
| fig5bc-words-information-falls-with-distance | reproduced |
| fig5b-counts-uninformative | not reproduced (counts slightly above zero) |
| fig5e-kl-divergence | reproduced |
| fig5-entropy | not reproduced (values differ; words higher than counts) |
| figs7cd-number-of-words | reproduced |
| figs7e-downsampled-words | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: words cover every 10 ms bin of each trial, without a
speed filter, which the Methods do not mention; the KL divergence sums over the words seen in both
trials, as the claims file says; the Neuropixels sessions' tracking comes from the CA1 file.

What was found about the dataset: nothing beyond notebook 01.